In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


# 1. Data Loading 

In [2]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# It looks like the input directory is '/kaggle/input/commpetitions/playground-series-s6e4', let's use that
data_path = '/kaggle/input/competitions/playground-series-s6e4/'

try:
    train_df = pd.read_csv(data_path + 'train.csv')
    test_df = pd.read_csv(data_path + 'test.csv')
    sample_submission_df = pd.read_csv(data_path + 'sample_submission.csv')
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Trying the local 'data' directory...")
    train_df = pd.read_csv('data/train.csv')
    test_df = pd.read_csv('data/test.csv')
    sample_submission_df = pd.read_csv('data/sample_submission.csv')

print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

Train data shape: (630000, 21)
Test data shape: (270000, 20)


# 2. Data Cleaning & Quality Check

In this section, we perform a basic data quality check by identifying missing values, duplicates, and examining the distribution of features to ensure there are no unexpected outliers or data entry errors.

In [3]:
print(f"Total missing values: {train_df.isnull().sum().sum()}")
print(f"Total duplicate rows: {train_df.duplicated().sum()}")
train_df.describe()

Total missing values: 0
Total duplicate rows: 0


# 3. Feature Engineering

Based on feature importance analysis, we add interactions between top variables like Rainfall, Temperature, and Soil Moisture.

In [ ]:
def create_features(df):
    # Evapotranspiration proxy
    df['Evapotranspiration_Proxy'] = (df['Temperature_C'] * df['Wind_Speed_kmh']) / (df['Humidity'] + 1)
    # Water balance proxy
    df['Total_Water_Added'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    # Mulching impact on moisture
    mulch_map = {'Yes': 1, 'No': 0}
    df['Moisture_Retention_Index'] = df['Soil_Moisture'] * df['Mulching_Used'].map(mulch_map)
    # Growth Stage impact
    stage_map = {'Sowing': 1, 'Vegetative': 2, 'Flowering': 3, 'Maturation': 4, 'Harvesting': 5}
    df['Growth_Water_Demand'] = df['Crop_Growth_Stage'].map(stage_map).fillna(0) * df['Soil_Moisture']
    
    return df

train_df = create_features(train_df)
test_df = create_features(test_df)
print("New features added. Shape:", train_df.shape)

# 4. Data Preprocessing

In [4]:
X = train_df.drop('Irrigation_Need', axis=1)
y = train_df['Irrigation_Need']

categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['number']).columns

for col in categorical_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    test_df[col] = le.transform(test_df[col])

scaler = StandardScaler()
numerical_features_no_id = [col for col in numerical_features if col != 'id']
X[numerical_features_no_id] = scaler.fit_transform(X[numerical_features_no_id])
test_df[numerical_features_no_id] = scaler.transform(test_df[numerical_features_no_id])

# 5. Model Training

## 5.1 Model Training Strategy

For this Kaggle submission, the LightGBM model is trained on the entire training dataset (`X` and `y`). This approach is common in Kaggle competitions to maximize the data available for the model to learn from, potentially leading to better performance on the unseen test set.

While a `train_test_split` was initially considered for local validation, it has been removed to ensure the final model leverages all training data.

### Further Improvements:

To potentially enhance model performance, consider these steps:

*   **Hyperparameter Tuning:** The current LightGBM model uses default parameters. Techniques like GridSearchCV or RandomizedSearchCV (from `sklearn.model_selection`) can be used to find optimal hyperparameters for LightGBM.
*   **Cross-Validation:** Implement k-fold cross-validation during hyperparameter tuning to get a more robust estimate of the model's performance and to prevent overfitting.
*   **Feature Engineering:** Create new features from existing ones (e.g., interaction terms, polynomial features) that might capture more complex relationships in the data.
*   **Ensemble Methods:** Combine predictions from multiple models (e.g., stacking, bagging, boosting with different algorithms) to often achieve higher accuracy than a single model.
*   **Handling Imbalanced Data:** If the target variable `Irrigation_Need` is imbalanced, consider techniques like SMOTE or class weighting to address it.

## 5.2 Hyperparameter Tuning 
To improve the model's performance, we can tune its hyperparameters. We will use `GridSearchCV` to find the best combination of hyperparameters from a predefined grid.

**Note:** This process can be computationally expensive and time-consuming, especially with a large dataset. For this example, we will use a small subset of the data and a small parameter grid.

In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

# Define a smaller subset of the data for faster tuning
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.1, stratify=y, random_state=42)
# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 50],
}

# Initialize the model
lgbm = lgb.LGBMClassifier(random_state=42, class_weight='balanced', force_row_wise=True)

# Initialize StratifiedKFold
cv = StratifiedKFold(n_splits=3)

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=lgbm, param_grid=param_grid, cv=cv, scoring='f1_weighted', n_jobs=-1, verbose=2)
# Fit GridSearchCV
# Note: This is commented out by default to prevent long run times.
# Uncomment the following lines to run the grid search.
grid_search.fit(X_sample, y_sample)
print("Best parameters found: ", grid_search.best_params_)
best_params = grid_search.best_params_
#best_params = {'learning_rate': 0.1, 'n_estimators': 200, 'num_leaves': 50}

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[LightGBM] [Info] Total Bins 2960
[LightGBM] [Info] Number of data points in the train set: 63000, number of used features: 20
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Best parameters found:  {'learning_rate': 0.1, 'n_estimators': 200, 'num_leaves': 50}


## 5.3 Model training

In [6]:
model = lgb.LGBMClassifier(random_state=42, class_weight='balanced', force_row_wise=True, **best_params)
# For Kaggle submission, it's common to train on the full training dataset.
model.fit(X, y)

[LightGBM] [Info] Total Bins 2960
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 20
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


## 5.4 Feature Importance Analysis

In [ ]:
feature_imp = pd.DataFrame({'Value': model.feature_importances_, 'Feature': X.columns})
plt.figure(figsize=(10, 8))
sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", ascending=False).head(20))
plt.title('LightGBM Top Features (Importance)')
plt.tight_layout()
plt.show()

# 6. Prediction and Submission

In [7]:
test_predictions = model.predict(test_df)

submission_df = pd.DataFrame({'id': test_df['id'], 'Irrigation_Need': test_predictions})
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
print(submission_df.head())

Submission file created successfully!
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
